In [4]:
import h5py
import numpy as np
import matplotlib.pyplot as plt

from sklearn.model_selection import train_test_split
from sklearn.metrics import confusion_matrix
from sklearn.metrics import classification_report
from sklearn.metrics import ConfusionMatrixDisplay

import tensorflow as tf
from tensorflow.keras import layers, models
from tensorflow.keras.callbacks import EarlyStopping

In [5]:
import h5py

subset_path = r"C:\Users\Ahmed\radioml_subset_100k.h5"

with h5py.File(subset_path, "r") as hf:
    X_small = hf["X"][:]
    Y_small = hf["Y"][:]
    Z_small = hf["Z"][:]

print(X_small.shape)
print(Y_small.shape)
print(Z_small.shape)

(100000, 1024, 2)
(100000, 24)
(100000, 1)


In [6]:
X_train, X_temp, Y_train, Y_temp, Z_train, Z_temp = train_test_split(
    X_small,
    Y_small,
    Z_small,
    test_size=0.3,
    random_state=42,
    stratify=np.argmax(Y_small, axis=1)
)

X_val, X_test, Y_val, Y_test, Z_val, Z_test = train_test_split(
    X_temp,
    Y_temp,
    Z_temp,
    test_size=0.5,
    random_state=42,
    stratify=np.argmax(Y_temp, axis=1)
)

print(X_train.shape)
print(X_val.shape)
print(X_test.shape)

(70000, 1024, 2)
(15000, 1024, 2)
(15000, 1024, 2)


In [7]:
mean = np.mean(X_train)
std = np.std(X_train)

X_train = (X_train - mean) / std
X_val = (X_val - mean) / std
X_test = (X_test - mean) / std

print(np.mean(X_train))
print(np.std(X_train))

7.193429e-09
1.0


In [8]:
np.save("X_test.npy", X_test)
np.save("Y_test.npy", Y_test)
np.save("X_val.npy", X_val)
np.save("Y_val.npy", Y_val)

In [9]:
import tensorflow as tf
from tensorflow.keras import layers, models

input_shape = (1024, 2)

model_bn = models.Sequential([

    layers.Input(shape=input_shape),

    layers.Conv1D(64, kernel_size=3),
    layers.BatchNormalization(),
    layers.Activation('relu'),
    layers.MaxPooling1D(2),

    layers.Conv1D(128, kernel_size=3),
    layers.BatchNormalization(),
    layers.Activation('relu'),
    layers.MaxPooling1D(2),

    layers.Conv1D(256, kernel_size=3),
    layers.BatchNormalization(),
    layers.Activation('relu'),

    layers.GlobalAveragePooling1D(),

    layers.Dense(128, activation='relu'),
    layers.Dropout(0.3),

    layers.Dense(24, activation='softmax')
])

model_bn.compile(
    optimizer='adam',
    loss='categorical_crossentropy',
    metrics=['accuracy']
)

model_bn.summary()

Model: "sequential_1"

┏━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━┓
┃ Layer (type)                    ┃ Output Shape           ┃       Param # ┃
┡━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━┩
│ conv1d_3 (Conv1D)               │ (None, 1022, 64)       │           448 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ batch_normalization_3           │ (None, 1022, 64)       │           256 │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ activation_3 (Activation)       │ (None, 1022, 64)       │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling1d_2 (MaxPooling1D)  │ (None, 511, 64)        │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv1d_4 (Conv1D)               │ (None, 509, 128)       │        24,704 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ batch_normalization_4           │ (None, 509, 128)       │           512 │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ activation_4 (Activation)       │ (None, 509, 128)       │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ max_pooling1d_3 (MaxPooling1D)  │ (None, 254, 128)       │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ conv1d_5 (Conv1D)               │ (None, 252, 256)       │        98,560 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ batch_normalization_5           │ (None, 252, 256)       │         1,024 │
│ (BatchNormalization)            │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ activation_5 (Activation)       │ (None, 252, 256)       │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ global_average_pooling1d_1      │ (None, 256)            │             0 │
│ (GlobalAveragePooling1D)        │                        │               │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_2 (Dense)                 │ (None, 128)            │        32,896 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dropout_1 (Dropout)             │ (None, 128)            │             0 │
├─────────────────────────────────┼────────────────────────┼───────────────┤
│ dense_3 (Dense)                 │ (None, 24)             │         3,096 │
└─────────────────────────────────┴────────────────────────┴───────────────┘

 Total params: 161,496 (630.84 KB)

 Trainable params: 160,600 (627.34 KB)

 Non-trainable params: 896 (3.50 KB)

In [10]:
from tensorflow.keras.callbacks import EarlyStopping, ModelCheckpoint

early_stop = EarlyStopping(
    monitor='val_loss',
    patience=5,
    restore_best_weights=True
)

checkpoint = ModelCheckpoint(
    "improved_v1_batchnorm.keras",
    monitor='val_loss',
    save_best_only=True
)

In [11]:
history_bn = model_bn.fit(
    X_train,
    Y_train,
    validation_data=(X_val, Y_val),
    epochs=40,
    batch_size=256,
    callbacks=[early_stop, checkpoint]
)

Epoch 1/40
274/274 ━━━━━━━━━━━━━━━━━━━━ 89s 319ms/step - accuracy: 0.2009 - loss: 2.5239 - val_accuracy: 0.2100 - val_loss: 2.7807
Epoch 2/40
274/274 ━━━━━━━━━━━━━━━━━━━━ 88s 320ms/step - accuracy: 0.2958 - loss: 2.0612 - val_accuracy: 0.1957 - val_loss: 4.1676
Epoch 3/40
274/274 ━━━━━━━━━━━━━━━━━━━━ 88s 319ms/step - accuracy: 0.3186 - loss: 1.9468 - val_accuracy: 0.2453 - val_loss: 2.4378
Epoch 4/40
274/274 ━━━━━━━━━━━━━━━━━━━━ 88s 321ms/step - accuracy: 0.3335 - loss: 1.8908 - val_accuracy: 0.3309 - val_loss: 1.9061
Epoch 5/40
274/274 ━━━━━━━━━━━━━━━━━━━━ 88s 322ms/step - accuracy: 0.3420 - loss: 1.8635 - val_accuracy: 0.2584 - val_loss: 2.6522
Epoch 6/40
274/274 ━━━━━━━━━━━━━━━━━━━━ 88s 322ms/step - accuracy: 0.3498 - loss: 1.8387 - val_accuracy: 0.3465 - val_loss: 1.8591
Epoch 7/40
274/274 ━━━━━━━━━━━━━━━━━━━━ 224s 820ms/step - accuracy: 0.3534 - loss: 1.8269 - val_accuracy: 0.2699 - val_loss: 2.6083
Epoch 8/40
274/274 ━━━━━━━━━━━━━━━━━━━━ 88s 322ms/step - accuracy: 0.3624 - loss: 

In [12]:
test_loss_bn, test_acc_bn = model_bn.evaluate(
    X_test,
    Y_test,
    verbose=1
)

print("BatchNorm Test Accuracy:", test_acc_bn)

469/469 ━━━━━━━━━━━━━━━━━━━━ 3s 7ms/step - accuracy: 0.3700 - loss: 1.7727
BatchNorm Test Accuracy: 0.3700000047683716
